# Clean & Split — Extended Features (CC1 + CC2 only)

**New notebook — does not touch any existing pipeline file.** Input:
`data/merged_extended/{complex_case1,complex_case2}_merged_extended.csv` (from
`merge_extended.ipynb`).

Combines what two separate stages did in the original lineage (`EDA/preprocessing.ipynb`'s
labeling/rate-computation/per-container normalization, and `clean_and_split.ipynb`'s
cleaning/splitting/RobustScaler) into one notebook, applied to the **11-feature**
extended set, for **CC1 and CC2 only** (matches the project's current CC2-only
drift scope).

**Extended feature set** (7 original + 4 new):
- `container_cpu_usage/system/user_seconds_rate` (existing, rate)
- `container_memory_usage/working_set_bytes`, `container_memory_rss`, `container_memory_cache` (existing, gauges)
- `container_cpu_cfs_throttled_seconds_rate`, `container_cpu_cfs_throttled_periods_rate` (**new** — direct CPU-throttling signal, converted from cumulative counters the same way as the existing CPU rate features)
- `container_threads` (**new** gauge — hypothesis: pod-failure recall, currently the weakest fault type, may show up more clearly here than in CPU/memory alone)
- `memory_limit_proximity` (**new**, derived = `memory_usage_bytes / spec_memory_limit_bytes` — "how close to the OOM ceiling", not just raw usage)

Same design decisions as the original pipeline, applied consistently: delay/loss
relabeled to normal, per-container min-max normalization per case, RobustScaler
fit on CC1-train only, CC1 split 70/10/20 by time, CC2 kept whole as the drift set.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
import joblib, json, os

BASE       = r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'
MERGED_DIR = os.path.join(BASE, 'experiments', 'models_extended', 'data', 'merged_extended')
GT_DIR     = os.path.join(BASE, 'data', 'raw')
OUT_DIR    = os.path.join(BASE, 'experiments', 'models_extended', 'data', 'processed_extended')
MODEL_DIR  = os.path.join(BASE, 'experiments', 'models_extended', 'model')
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

CASES = {
    'complex_case1': {'merged': 'complex_case1_merged_extended.csv',
                       'gt': os.path.join(GT_DIR, 'Complex Case-1', 'Case-1', 'groundtruth', 'groundtruth.json')},
    'complex_case2': {'merged': 'complex_case2_merged_extended.csv',
                       'gt': os.path.join(GT_DIR, 'Complex Case-2', 'Case-2', 'groundtruth', 'groundtruth.json')},
}

CUMULATIVE_COLS = {
    'container_cpu_usage_seconds_total':          'container_cpu_usage_seconds_rate',
    'container_cpu_system_seconds_total':         'container_cpu_system_seconds_rate',
    'container_cpu_user_seconds_total':           'container_cpu_user_seconds_rate',
    'container_cpu_cfs_throttled_seconds_total':  'container_cpu_cfs_throttled_seconds_rate',
    'container_cpu_cfs_throttled_periods_total':  'container_cpu_cfs_throttled_periods_rate',
}
GAUGE_COLS = [
    'container_memory_usage_bytes', 'container_memory_working_set_bytes',
    'container_memory_rss', 'container_memory_cache', 'container_threads',
]
FEATURE_COLS = list(CUMULATIVE_COLS.values()) + GAUGE_COLS + ['memory_limit_proximity']

EXCLUDED_FAULT_TYPES = ['delay', 'loss']
GAP_THRESH_SEC = 30
TRAIN_FRAC, VAL_FRAC = 0.70, 0.80
CLIP = 20.0

print('FEATURE_COLS (11 total):', FEATURE_COLS)

FEATURE_COLS (11 total): ['container_cpu_usage_seconds_rate', 'container_cpu_system_seconds_rate', 'container_cpu_user_seconds_rate', 'container_cpu_cfs_throttled_seconds_rate', 'container_cpu_cfs_throttled_periods_rate', 'container_memory_usage_bytes', 'container_memory_working_set_bytes', 'container_memory_rss', 'container_memory_cache', 'container_threads', 'memory_limit_proximity']


## Step 1 — Load, label, clean each case independently

In [2]:
def apply_labels(df, gt):
    df = df.copy()
    df['short_id'] = df['cmdb_id'].str.replace('observe.', '', regex=False)
    df['label'] = 0
    df['failure_type'] = None
    for ts, cid, dur, ftype in zip(gt['timestamp'], gt['cmdb_id'], gt['duration'], gt['failure_type']):
        in_window = (df['timestamp'] >= ts) & (df['timestamp'] < ts + dur)
        id_match  = df['short_id'] == cid
        mask = in_window & id_match
        df.loc[mask, 'label'] = 1
        df.loc[mask, 'failure_type'] = ftype
    return df.drop(columns=['short_id'])


processed = {}
for case, paths in CASES.items():
    df = pd.read_csv(os.path.join(MERGED_DIR, paths['merged']), low_memory=False)
    with open(paths['gt']) as f:
        gt = json.load(f)
    df = apply_labels(df, gt)

    # forward/backward-fill the small number of missing values in the new columns (per container)
    df = df.sort_values(['cmdb_id', 'timestamp']).reset_index(drop=True)
    fill_cols = list(CUMULATIVE_COLS.keys()) + ['container_threads', 'container_spec_memory_limit_bytes']
    df[fill_cols] = df.groupby('cmdb_id')[fill_cols].transform(lambda x: x.ffill().bfill())

    # dedup (safety net — merge_extended.ipynb already deduped each source table before joining)
    before = len(df)
    df = df.drop_duplicates(subset=['cmdb_id', 'timestamp'], keep='first').reset_index(drop=True)

    # gap marking
    df['time_diff'] = df.groupby('cmdb_id')['timestamp'].diff().fillna(15.0)
    df['is_gap'] = df['time_diff'] > GAP_THRESH_SEC

    n_anom = int(df['label'].sum())
    print(f'{case}: {before:,} -> {len(df):,} rows (dedup safety net)  |  anomalies: {n_anom:,}  '
          f'({n_anom/len(df)*100:.3f}%)  types={df.loc[df.label==1,"failure_type"].value_counts().to_dict()}')
    processed[case] = df

complex_case1: 223,830 -> 223,830 rows (dedup safety net)  |  anomalies: 480  (0.214%)  types={'delay': 124, 'loss': 100, 'pod-failure': 92, 'cpu': 88, 'memory': 76}
complex_case2: 77,787 -> 77,787 rows (dedup safety net)  |  anomalies: 812  (1.044%)  types={'loss': 360, 'memory': 240, 'cpu': 120, 'delay': 80, 'pod-failure': 12}


## Step 2 — Rate features for cumulative counters (existing 3 + new 2 throttled)

Same treatment as the original pipeline: `diff(value) / diff(timestamp)`,
clipped at 0 to remove counter-reset artifacts, first row per container dropped
(no previous value to diff against).

In [3]:
for case, df in processed.items():
    df = df.sort_values(['cmdb_id', 'timestamp']).copy()
    grp = df.groupby('cmdb_id')
    for total_col, rate_col in CUMULATIVE_COLS.items():
        val_delta = grp[total_col].diff()
        t_delta   = grp['timestamp'].diff()
        df[rate_col] = np.where(t_delta == 0, 0.0, (val_delta / t_delta).clip(lower=0))
    before = len(df)
    df = df.dropna(subset=list(CUMULATIVE_COLS.values()))  # first row per container: no diff available
    print(f'{case}: dropped {before - len(df)} rows (first row per container, expected 27)')
    processed[case] = df

complex_case1: dropped 27 rows (first row per container, expected 27)
complex_case2: dropped 27 rows (first row per container, expected 27)


## Step 3 — Derive `memory_limit_proximity`

`container_memory_usage_bytes / container_spec_memory_limit_bytes` — proximity
to the container's own configured memory ceiling, not just raw usage in bytes.

In [4]:
for case, df in processed.items():
    df['memory_limit_proximity'] = df['container_memory_usage_bytes'] / df['container_spec_memory_limit_bytes']
    print(f'{case}: memory_limit_proximity range = [{df["memory_limit_proximity"].min():.4f}, {df["memory_limit_proximity"].max():.4f}]')
    processed[case] = df

complex_case1: memory_limit_proximity range = [0.0000, 0.1531]
complex_case2: memory_limit_proximity range = [0.0116, 0.1475]


C:\Users\jthar\AppData\Local\Temp\ipykernel_37928\2423771932.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['memory_limit_proximity'] = df['container_memory_usage_bytes'] / df['container_spec_memory_limit_bytes']


## Step 4 — Relabel delay/loss to normal (same decision as the main pipeline)

In [5]:
for case, df in processed.items():
    excl_mask = df['failure_type'].isin(EXCLUDED_FAULT_TYPES)
    print(f'{case}: relabeling {int(excl_mask.sum())} delay/loss rows to normal')
    df.loc[excl_mask, 'label'] = 0
    df.loc[excl_mask, 'failure_type'] = None
    processed[case] = df

complex_case1: relabeling 224 delay/loss rows to normal
complex_case2: relabeling 440 delay/loss rows to normal


## Step 5 — Per-container min-max normalization (per case, matching the original convention)

In [6]:
def normalize_per_container(df, feature_cols):
    df = df.copy()
    for col in feature_cols:
        g = df.groupby('cmdb_id')[col]
        c_min, c_max = g.transform('min'), g.transform('max')
        denom = (c_max - c_min).replace(0, 1)
        df[col] = (df[col] - c_min) / denom
    return df

for case, df in processed.items():
    processed[case] = normalize_per_container(df, FEATURE_COLS)
print('Per-container min-max normalization done for both cases.')

Per-container min-max normalization done for both cases.


## Step 6 — CC1 split (70/10/20 by time) + CC2 as the sole drift set

In [7]:
cc1 = processed['complex_case1']
normals = cc1[cc1['label'] == 0]
t1 = normals['timestamp'].quantile(TRAIN_FRAC)
t2 = normals['timestamp'].quantile(VAL_FRAC)

cc1_train = normals[normals['timestamp'] < t1].copy()
cc1_val   = normals[(normals['timestamp'] >= t1) & (normals['timestamp'] < t2)].copy()
cc1_test  = pd.concat([normals[normals['timestamp'] >= t2], cc1[cc1['label'] == 1]]).sort_values(['cmdb_id', 'timestamp']).copy()

drift_cc2 = processed['complex_case2'].copy()

print(f'cc1_train: {len(cc1_train):,}  cc1_val: {len(cc1_val):,}  cc1_test: {len(cc1_test):,} ({(cc1_test["label"]==1).sum()} anomalies)')
print(f'drift_cc2: {len(drift_cc2):,} ({(drift_cc2["label"]==1).sum()} anomalies)')

cc1_train: 156,479  cc1_val: 22,356  cc1_test: 44,968 (256 anomalies)
drift_cc2: 77,760 (372 anomalies)


## Step 7 — RobustScaler fit on CC1-train only, applied everywhere

In [8]:
scaler = RobustScaler()
scaler.fit(cc1_train[FEATURE_COLS])
print('center_:', np.round(scaler.center_, 4))
print('scale_ :', np.round(scaler.scale_, 4))

def scale_and_clip(df):
    out = df.copy()
    out[FEATURE_COLS] = np.clip(scaler.transform(out[FEATURE_COLS]), -CLIP, CLIP)
    return out

cc1_train = scale_and_clip(cc1_train)
cc1_val   = scale_and_clip(cc1_val)
cc1_test  = scale_and_clip(cc1_test)
drift_cc2 = scale_and_clip(drift_cc2)
print('Scaling applied.')

center_: [0.     0.     0.     0.     0.     0.7934 0.7924 0.762  0.4545 0.9778
 0.7934]
scale_ : [1.     1.     1.     1.     1.     0.4476 0.4528 0.4531 0.4484 0.1818
 0.4476]
Scaling applied.


## Step 8 — Save

In [9]:
SAVE_COLS = ['timestamp', 'cmdb_id'] + FEATURE_COLS + ['label', 'failure_type', 'time_diff', 'is_gap']
splits = {'cc1_train': cc1_train, 'cc1_val': cc1_val, 'cc1_test': cc1_test, 'drift_cc2': drift_cc2}
for name, df in splits.items():
    path = os.path.join(OUT_DIR, f'{name}.csv')
    df[SAVE_COLS].to_csv(path, index=False)
    print(f'  {name:10s} -> {path}  ({len(df):,} rows)')

joblib.dump({'scaler': scaler, 'feature_cols': FEATURE_COLS, 'clip': CLIP}, os.path.join(MODEL_DIR, 'cc1_scaler_extended.pkl'))
print('Scaler saved.')

  cc1_train  -> c:\Users\jthar\Documents\Claude\Projects\module3\data\processed_extended\cc1_train.csv  (156,479 rows)
  cc1_val    -> c:\Users\jthar\Documents\Claude\Projects\module3\data\processed_extended\cc1_val.csv  (22,356 rows)
  cc1_test   -> c:\Users\jthar\Documents\Claude\Projects\module3\data\processed_extended\cc1_test.csv  (44,968 rows)
  drift_cc2  -> c:\Users\jthar\Documents\Claude\Projects\module3\data\processed_extended\drift_cc2.csv  (77,760 rows)
Scaler saved.


## Summary

| Output | Purpose |
|---|---|
| `data/processed_extended/{cc1_train,cc1_val,cc1_test,drift_cc2}.csv` | 11-feature version of the same splits |
| `models_extended/cc1_scaler_extended.pkl` | RobustScaler fit on extended CC1-train |

Next: `windowing_pca_extended.ipynb` builds sliding windows + PCA on these
11 features (up from 7 → 330-dim flattened windows, up from 210).